# PersonaLM: dataset recipe -> tokens -> pretrain `s30m` on a free Colab GPU

Public text only. Your private chats are NOT used here.
This notebook runs the same code as `python -m persona_lm hfdata ...` on your laptop: it reads a **dataset recipe** (a small JSON file made by `hfdata new`), streams the Hugging Face dataset (never the whole 30 GB), filters + cleans it, makes token files with your frozen `tok.json`, and trains `s30m`.

**Before running:** Runtime -> Change runtime type -> **T4 GPU**. Put the Colab kit (`hfdata colab-kit ID` makes it) into `MyDrive/persona/`.
**After a disconnect:** just run all the cells again. Shards, token files and checkpoints live in Google Drive, so every step continues where it stopped.

In [ ]:
# ---- settings (the only cell you need to edit) ----
RECIPE_ID = "oshichats"        # the recipe in MyDrive/persona/dataset_recipes/<id>.json
DRIVE_DIR = "/content/drive/MyDrive/persona"
FILTER_OVERRIDES = {}          # change a number filter without editing the file, e.g. {"score": 2.5}. Run the "peek" cell first to see sensible values
MAX_KEPT = None                # None = the recipe's number (oshichats: 5,000,000 messages)
RESTART_SCAN = False           # True = throw the saved shards away (they are MOVED to _to_delete) and scan again. Needed after changing a filter
ALSO = []                      # more text to mix in, relative to DRIVE_DIR, e.g. ["public_mix/tl.jsonl", "public_mix/ceb.jsonl"] (keeps Tagalog/Cebuano alive)
SIZE = "s30m"
CTX = 4096                     # window in tokens
PASSES = 2.0                   # how many times to read the new tokens; STEPS is worked out from the token count
BATCH = 2                      # micro-batch. If you get "out of memory": BATCH = 1 and ACCUM = 4
ACCUM = 2                      # BATCH x ACCUM should stay 4 so every step is 16,384 tokens
PRECISION = "auto"             # "auto" = fp16 on a T4. If the loss turns into nan, use "fp32" (slower)
LR = 0.0002                    # learning rate when CONTINUING from the pretrained model (lower than a fresh pretrain, so Tagalog/Cebuano are not forgotten)
MIN_LR = 0.00002
WARMUP = 100
BASE_CKPT = "base_ckpt.npz"    # your pretrained s30m in MyDrive/persona/ (copy work3\run_pre\ckpt.npz). Missing = a fresh run from random weights
MAX_HOURS = 10.0               # safety stop: saves a checkpoint and ends cleanly before Colab's session limit

In [ ]:
# ---- connect Google Drive, copy the package, install `datasets` ----
import glob, json, os, shutil, signal, subprocess, sys, time
from google.colab import drive
drive.mount("/content/drive")
P = DRIVE_DIR
for need in ("persona_lm/__init__.py", "dataset_recipes/%s.json" % RECIPE_ID, "tok.json"):
    assert os.path.exists(os.path.join(P, need)), "missing in Drive: %s/%s (unzip the Colab kit into MyDrive)" % (P, need)
CODE = "/content/code"
shutil.rmtree(CODE, ignore_errors=True)
os.makedirs(CODE)
shutil.copytree(os.path.join(P, "persona_lm"), os.path.join(CODE, "persona_lm"), ignore=shutil.ignore_patterns("__pycache__"))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "datasets"], check=True)
TOK = os.path.join(P, "tok.json")
env = dict(os.environ, PYTHONPATH=CODE, PYTHONUNBUFFERED="1", PERSONA_PLAIN="1")

def run(args, limit=None):
    """Runs `python -m persona_lm ...` inside the Drive project folder and prints its output as it comes."""
    cmd = [sys.executable, "-m", "persona_lm"] + args
    if limit:
        cmd = ["timeout", "-s", "INT", str(int(limit))] + cmd
    print("$", " ".join(cmd), flush=True)
    p = subprocess.Popen(cmd, cwd=P, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    try:
        for line in p.stdout:
            print(line, end="")
    except KeyboardInterrupt:                  # the stop button in Colab: ask the program to save and stop, then wait for it
        p.send_signal(signal.SIGINT)
        for line in p.stdout:
            print(line, end="")
    return p.wait()

# apply the optional overrides to the recipe file (only when something really changes)
rp = os.path.join(P, "dataset_recipes", RECIPE_ID + ".json")
with open(rp, encoding="utf-8") as f:
    rec = json.load(f)
changed = False
for flt in rec["filters"]:
    if flt["field"] in FILTER_OVERRIDES and flt["value"] != FILTER_OVERRIDES[flt["field"]]:
        flt["value"] = FILTER_OVERRIDES[flt["field"]]
        changed = True
if MAX_KEPT and rec["max_kept"] != MAX_KEPT:
    rec["max_kept"] = MAX_KEPT
    changed = True
if changed:
    with open(rp + ".tmp", "w", encoding="utf-8") as f:
        json.dump(rec, f, indent=2, ensure_ascii=False)
    os.replace(rp + ".tmp", rp)
    print("recipe updated:", [(x["field"], x["op"], x["value"]) for x in rec["filters"]], "max_kept", rec["max_kept"])
print("recipe:", RECIPE_ID, "|", rec["hf_repo"], "| licence:", rec.get("license") or "unknown", "| private training only" if rec.get("private_only") else "")

In [ ]:
# ---- peek: the first 20,000 rows. Look at the quantiles of the number filters (score) and the sample messages, then set FILTER_OVERRIDES above if needed ----
run(["hfdata", "peek", RECIPE_ID, "--rows", "20000"])

In [ ]:
# ---- stream + filter + clean (can take a while; it saves shards in Drive, so a disconnect loses at most one shard) ----
args = ["hfdata", "run", RECIPE_ID, "--yes", "--plain"]
if RESTART_SCAN:
    args.append("--restart")
rc = run(args)
print("exit code", rc, "(0 = finished; run this cell again to continue after a stop)")

In [ ]:
# ---- make token files with your frozen tokenizer (skipped when the folder is already finished) ----
DATA_DRIVE = os.path.join(P, "data_pre_" + RECIPE_ID)
args = ["hfdata", "tokens", RECIPE_ID, "--tokenizer", TOK, "--out", DATA_DRIVE]
if ALSO:
    args += ["--also"] + [os.path.join(P, x) for x in ALSO]
rc = run(args)
assert os.path.exists(os.path.join(DATA_DRIVE, "meta.json")), "token files were not made; read the messages above"
with open(os.path.join(DATA_DRIVE, "meta.json")) as f:
    meta = json.load(f)
STEPS = max(1, int(PASSES * meta["train_tokens"] / (BATCH * ACCUM * CTX)))
print("train tokens: {:,}   steps for {} passes: {:,}".format(meta["train_tokens"], PASSES, STEPS))

In [ ]:
# ---- copy the tokens to the Colab disk (training reads them many times; Drive is slow for that), then report ----
DATA = "/content/data_local"
if not os.path.exists(os.path.join(DATA, "meta.json")):
    shutil.rmtree(DATA, ignore_errors=True)
    shutil.copytree(DATA_DRIVE, DATA)
run(["data-report", "--data-dir", DATA, "--size", SIZE, "--ctx", str(CTX), "--batch", str(BATCH), "--accum", str(ACCUM), "--steps", str(STEPS), "--tokenizer", TOK])
subprocess.run("nvidia-smi", shell=True)

In [ ]:
# ---- is the GPU maths right? (about a minute). Stop here if it says MISMATCH ----
rc = run(["check-gpu"])
print("check-gpu exit code:", rc)

In [ ]:
# ---- training: the run folder is in Drive, so ckpt.npz survives a disconnect. Run this cell again and it resumes by itself ----
RUN = os.path.join(P, "run_%s_%s" % (RECIPE_ID, SIZE))
os.makedirs(RUN, exist_ok=True)
args = ["train", "--run", RUN, "--data-dir", DATA, "--size", SIZE, "--ctx", str(CTX), "--steps", str(STEPS), "--batch", str(BATCH),
        "--accum", str(ACCUM), "--precision", PRECISION, "--lr", str(LR), "--min-lr", str(MIN_LR), "--warmup", str(WARMUP), "--plain"]
base = os.path.join(P, BASE_CKPT)
if not os.path.exists(os.path.join(RUN, "ckpt.npz")):
    if os.path.exists(base):
        args += ["--init-from", base]
        print("NEW run, starting from your pretrained model:", base)
    else:
        print("NEW run from random weights (no %s in Drive)" % BASE_CKPT)
else:
    print("resuming the run in", RUN)
t0 = time.time()
rc = run(args, limit=MAX_HOURS * 3600)
print("exit code", rc, "after %.2f h" % ((time.time() - t0) / 3600))

In [ ]:
# ---- result ----
ck = os.path.join(RUN, "ckpt.npz")
sys.path.insert(0, CODE)
from persona_lm.report import read_ckpt_meta
m = read_ckpt_meta(ck) or {}
print("checkpoint:", ck, "%.0f MB" % (os.path.getsize(ck) / 1e6))
print("step:", m.get("step"), "of", STEPS)
print("DONE" if m.get("step", 0) >= STEPS else "NOT finished yet: run the training cell again; it continues.")

In [ ]:
# ---- sample chat lines from the checkpoint (a pretrained model continues text; it is not a chat assistant yet) ----
PROMPTS = ["hello chat, how is everyone doing today?\n", "[streamer] that was so funny\n", "what game are we playing\n"]
for pr in PROMPTS:
    run(["generate", "--ckpt", ck, "--tokenizer", TOK, "--prompt", pr, "--max-new", "80", "--temperature", "0.8", "--top-k", "40"])